#  ETL-Pipeline

In [1]:
import sqlalchemy
import pymysql

In [2]:
import pandas as pd
import requests
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from lat_lon_parser import parse
import os
from dotenv import load_dotenv

## Web Scrapping for City Data

In [3]:
# First look at the data from Berlin alone
url = "https://en.wikipedia.org/wiki/Berlin"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)

soup_Berlin = BeautifulSoup(response.content, 'html.parser')


In [4]:
# exploring the data
soup_Berlin.prettify()

'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="en">\n <head>\n  <meta charset="utf-8"/>\n  <title>\n   Berlin - Wikipedia\n  </title>\n  <script>\n   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-p

In [5]:
# looking at the infobox
soup_Berlin.find_all(class_="infobox-data")

[<td class="infobox-data"><span class="flagicon nowrap"><span class="mw-image-border" typeof="mw:File"><span><img alt="" class="mw-file-element" data-file-height="600" data-file-type="drawing" data-file-width="1000" decoding="async" height="14" loading="lazy" resource="https://en.wikipedia.org/wiki/File:Flag_of_Germany.svg" src="//thumb.wikimedia.org/wikipedia/en/thumb/b/ba/Flag_of_Germany.svg/40px-Flag_of_Germany.svg.png?utm_source=en.wikipedia.org&amp;utm_campaign=parser&amp;utm_content=thumbnail" srcset="//thumb.wikimedia.org/wikipedia/en/thumb/b/ba/Flag_of_Germany.svg/60px-Flag_of_Germany.svg.png?utm_source=en.wikipedia.org&amp;utm_campaign=parser&amp;utm_content=thumbnail 2x" width="23"/></span></span> </span><a href="https://en.wikipedia.org/wiki/Germany" rel="mw:WikiLink" title="Germany">Germany</a></td>,
 <td class="infobox-data"><a class="mw-redirect" href="https://en.wikipedia.org/wiki/Boroughs_and_neighborhoods_of_Berlin" rel="mw:WikiLink" title="Boroughs and neighborhoods o

In [6]:
# get country from infobox
soup_Berlin.find(class_="infobox-data").get_text()

' Germany'

In [7]:
berlin_country = soup_Berlin.find(class_="infobox-data").get_text()
berlin_country

' Germany'

In [8]:
# Berlin Latitude
berlin_latitude = soup_Berlin.find(class_="latitude").get_text()
berlin_latitude

'52°31′12″N'

In [9]:
# Berlin Longitude
berlin_longitude = soup_Berlin.find(class_="longitude").get_text()
berlin_longitude

'13°24′18″E'

Second city Hamburg

In [10]:
url = 'https://www.wikipedia.org/wiki/Hamburg'
headers = {'User-Agent': 'Chrome/139.0.0.0'}

response = requests.get(url, headers=headers)
hamburg_soup = BeautifulSoup(response.content, 'html.parser')

hamburg_country = hamburg_soup.find(class_="infobox-data").get_text()
hamburg_latitude = hamburg_soup.find(class_="latitude").get_text()
hamburg_longitude = hamburg_soup.find(class_="longitude").get_text()

hamburg_country, hamburg_latitude, hamburg_longitude

(' Germany', '53°33′N', '10°00′E')

Third city Cologne

In [11]:
url = 'https://en.wikipedia.org/wiki/Cologne'
headers = {'User-Agent': 'Chrome/139.0.0.0'}

response = requests.get(url, headers=headers)
cologne_soup = BeautifulSoup(response.content, 'html.parser')

cologne_country = cologne_soup.find(class_="infobox-data").get_text()
cologne_latitude = cologne_soup.find(class_="latitude").get_text()
cologne_longitude = cologne_soup.find(class_="longitude").get_text()

cologne_country, cologne_latitude, cologne_longitude


('Germany', '50°56′11″N', '6°57′10″E')

Get all city data with a for loop

In [12]:
cities = ["Berlin", "Hamburg", "Cologne"]

countries = []
latitudes = []
longitudes = []

for city in cities:
  # get the soup for the city
  url = f"https://www.wikipedia.org/wiki/{city}"
  headers = {'User-Agent': 'Chrome/134.0.0.0'}

  response = requests.get(url, headers=headers)
  city_soup = BeautifulSoup(response.content, 'html.parser')

  # extract the data
  city_country = city_soup.find(class_="infobox-data").get_text()
  city_latitude = city_soup.find(class_="latitude").get_text()
  city_longitude = city_soup.find(class_="longitude").get_text()

  # append data to a list
  countries.append(city_country)
  latitudes.append(city_latitude)
  longitudes.append(city_longitude)

In [13]:
print(f"The cities are in the following countries: {countries}")
print(f"The cities have the following latitudes: {latitudes}")
print(f"The cities have the following longitudes: {longitudes}")

The cities are in the following countries: [' Germany', ' Germany', 'Germany']
The cities have the following latitudes: ['52°31′12″N', '53°33′N', '50°56′11″N']
The cities have the following longitudes: ['13°24′18″E', '10°00′E', '6°57′10″E']


Creating a DataFrame

In [14]:
cities_df = pd.DataFrame({"City": cities,
                          "Country": countries,
                          "Latitude": latitudes,
                          "Longitude": longitudes})

cities_df

,City,Country,Latitude,Longitude
0,Berlin,Germany,52°31′12″N,13°24′18″E
1,Hamburg,Germany,53°33′N,10°00′E
2,Cologne,Germany,50°56′11″N,6°57′10″E


For easier analysis parse the latitude and longitude

In [15]:
# example
parse(berlin_latitude)

52.519999999999996

Create a function to easily gather city data

In [16]:
def cities_dataframe(cities):

  city_data = []

  for city in cities:
    url = f"https://www.wikipedia.org/wiki/{city}"
    headers = {'User-Agent': 'Chrome/134.0.0.0'}

    response = requests.get(url, headers=headers)
    city_soup = BeautifulSoup(response.content, 'html.parser')

    # extract the relevant information
    city_latitude = city_soup.find(class_="latitude").get_text()
    city_longitude = city_soup.find(class_="longitude").get_text()
    country = city_soup.find(class_="infobox-data").get_text()

    # keep track of data per city
    city_data.append({"City": city,
                    "Country": country,
                    "Latitude": parse(city_latitude), # latitude in decimal format
                    "Longitude": parse(city_longitude), # longitude in decimal format
                    })

  return pd.DataFrame(city_data)

In [17]:
# call the function
list_of_cities = ["Berlin", "Hamburg", "Cologne"]

cities_df = cities_dataframe(list_of_cities)
cities_df

,City,Country,Latitude,Longitude
0,Berlin,Germany,52.520000,13.405000
1,Hamburg,Germany,53.550000,10.000000
2,Cologne,Germany,50.936389,6.952778


Gather city population

In [18]:
# search for the number for population
soup_Berlin.find(string="Population").find_next()
# The number for population is not next to the word "Population" on Wikipedia, need to refine the search.

<div class="ib-settlement-fn"><span class="nowrap"><span typeof="mw:Entity"> </span></span>(2022 census)<sup about="#mwt26" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-2"}}' id="cite_ref-2" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-2" id="mwEw"><span class="mw-reflink-text" id="mwFA"><span class="cite-bracket" id="mwFQ">[</span>2<span class="cite-bracket" id="mwFg">]</span></span></a></sup></div>

In [19]:
# finding the number
soup_Berlin.find(string="Population").find_next("td")

<td class="infobox-data">3,596,999</td>

In [20]:
# get string for population number
berlin_population = soup_Berlin.find(string="Population").find_next("td").get_text()
berlin_population

'3,596,999'

In [21]:
# remove commas from the number
berlin_population_clean = berlin_population.replace(",", "")
berlin_population_clean

'3596999'

In [22]:
# turn the number into an integer
berlin_population_clean = int(berlin_population_clean)
berlin_population_clean

3596999

In [23]:
cities = ["Berlin", "Hamburg", "Cologne"]

# create one single list to keep track of all values (instead of having separate lists)
population_data = []

for city in cities:
    url = f"https://www.wikipedia.org/wiki/{city}"
    headers = {'User-Agent': 'Chrome/134.0.0.0'}

    response = requests.get(url, headers=headers)
    city_soup = BeautifulSoup(response.content, 'html.parser')

    # extract the relevant information
    city_population = city_soup.find(string="Population").find_next("td").get_text()
    city_population_clean = city_population.replace(",", "")
    today = datetime.today().strftime("%d.%m.%Y")

    # for each city we append a dictionary of values to the list
    population_data.append({"City": city,
                     "population": int(city_population_clean),
                     "population_Timestamp": today
                    })

population_data = pd.DataFrame(population_data)
population_data

,City,population,population_Timestamp
0,Berlin,3596999,26.09.2026
1,Hamburg,1973896,26.09.2026
2,Cologne,1025523,26.09.2026


In [24]:
# Gather population from cities function:

def populations_dataframe(cities):

    population_data = []

    for city in cities:
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # extract the relevant information
        city_population = city_soup.find(string="Population").find_next("td").get_text()
        city_population_clean = int(city_population.replace(",", ""))
        today = datetime.today().strftime("%Y-%m-%d")

        # keep track of data per city
        population_data.append({"population": city_population_clean,
                        "timestamp_population": today
                        })

    return pd.DataFrame(population_data)

In [25]:
# call the populations function
cities = ["Berlin", "Cologne", "Hamburg"]

population_df = populations_dataframe(cities)
population_df

,population,timestamp_population
0,3596999,2026-09-26
1,1025523,2026-09-26
2,1973896,2026-09-26


## Python to SQL

### Matching tables in SQL
```sql
-- Drop the database if it already exists
DROP DATABASE IF EXISTS sql_ETL_cities;

-- Create the database
CREATE DATABASE sql_ETL_cities;

-- Use the database
USE sql_ETL_cities;

-- Create the 'cities' table
CREATE TABLE cities (
    city_id INT AUTO_INCREMENT, -- Automatically generated ID for each city
    city VARCHAR(255) NOT NULL, -- Name of the city
    latitude FLOAT, -- Latitude of the city
    longitude FLOAT, -- Longitude of the city
    country VARCHAR(255), -- Name of the city's country
    PRIMARY KEY (city_id) -- Primary key to uniquely identify each city
);

-- Create the 'populations' table
CREATE TABLE populations (
    city_id INT AUTO_INCREMENT, -- Automatically generated ID for each city
	population INT, -- number for population
    timestamp_population DATE, -- When the data was retrieved
    PRIMARY KEY (city_id), -- Primary key to uniquely identify each book
    FOREIGN KEY (city_id) REFERENCES cities(city_id) -- Foreign key to connect the population to the cities
);
```

In [26]:
# connecting to SQL
# password within .env file
load_dotenv()

schema = os.environ["schema_name"]
host = os.environ["host_address"]
user = os.environ["user_name"]
password = os.environ["password"]
port = os.environ["port_number"]

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

Connect cities_df with SQL and then load it.

In [27]:
cities_df.to_sql('cities',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3

In [28]:
cities_from_sql = pd.read_sql("cities", con=connection_string)
cities_from_sql

,city_id,city,latitude,longitude,country
0,1,Berlin,52.5200,13.40500,Germany
1,2,Hamburg,53.5500,10.00000,Germany
2,3,Cologne,50.9364,6.95278,Germany


Connect population_df with SQL and then load it.

In [29]:
population_df.to_sql('populations',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3

In [30]:
population_from_sql = pd.read_sql("populations", con=connection_string)
population_from_sql

,population,timestamp_population
0,3596999,2026-09-26
1,1025523,2026-09-26
2,1973896,2026-09-26


## Gathering Data from OpenWeather
Using the 5 day weather forecast (5 day / 3 hour forecast data) API

In [31]:
# Example gathering data for Berlin
load_dotenv()
latitude = 52.520008
longitude = 13.404954

API_key = os.environ["openWeatherApi"]

# check out the docs for more info on making an api call https://openweathermap.org/forecast5
url = (f"https://api.openweathermap.org/data/2.5/forecast?lat={latitude}&lon={longitude}&appid={API_key}&units=metric")

response = requests.get(url)
json = response.json()

json

{'cod': '200',
 'message': 0,
 'cnt': 40,
 'list': [{'dt': 1790445600,
   'main': {'temp': 18.77,
    'feels_like': 18.3,
    'temp_min': 16.88,
    'temp_max': 18.77,
    'pressure': 1022,
    'sea_level': 1022,
    'grnd_level': 1018,
    'humidity': 61,
    'temp_kf': 1.89,
    'dew_point': 11.1},
   'weather': [{'id': 804,
     'main': 'Clouds',
     'description': 'overcast clouds',
     'icon': '04n'}],
   'clouds': {'all': 92},
   'wind': {'speed': 1.37, 'deg': 336, 'gust': 2.48},
   'visibility': 10000,
   'pop': 0,
   'sys': {'pod': 'n'},
   'dt_txt': '2026-09-26 18:00:00'},
  {'dt': 1790456400,
   'main': {'temp': 13.96,
    'feels_like': 13.32,
    'temp_min': 11.08,
    'temp_max': 13.96,
    'pressure': 1023,
    'sea_level': 1023,
    'grnd_level': 1019,
    'humidity': 73,
    'temp_kf': 2.88,
    'dew_point': 9.2},
   'weather': [{'id': 803,
     'main': 'Clouds',
     'description': 'broken clouds',
     'icon': '04n'}],
   'clouds': {'all': 62},
   'wind': {'speed': 1

In [32]:
# output is a dictionary
json.keys()

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])

In [33]:
json['cod']

'200'

In [34]:
json['message']

0

In [35]:
json['cnt']

40

In [36]:
json['list']

[{'dt': 1790445600,
  'main': {'temp': 18.77,
   'feels_like': 18.3,
   'temp_min': 16.88,
   'temp_max': 18.77,
   'pressure': 1022,
   'sea_level': 1022,
   'grnd_level': 1018,
   'humidity': 61,
   'temp_kf': 1.89,
   'dew_point': 11.1},
  'weather': [{'id': 804,
    'main': 'Clouds',
    'description': 'overcast clouds',
    'icon': '04n'}],
  'clouds': {'all': 92},
  'wind': {'speed': 1.37, 'deg': 336, 'gust': 2.48},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'n'},
  'dt_txt': '2026-09-26 18:00:00'},
 {'dt': 1790456400,
  'main': {'temp': 13.96,
   'feels_like': 13.32,
   'temp_min': 11.08,
   'temp_max': 13.96,
   'pressure': 1023,
   'sea_level': 1023,
   'grnd_level': 1019,
   'humidity': 73,
   'temp_kf': 2.88,
   'dew_point': 9.2},
  'weather': [{'id': 803,
    'main': 'Clouds',
    'description': 'broken clouds',
    'icon': '04n'}],
  'clouds': {'all': 62},
  'wind': {'speed': 1.79, 'deg': 31, 'gust': 3.31},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'n'},

In [37]:
json['list'][0]

{'dt': 1790445600,
 'main': {'temp': 18.77,
  'feels_like': 18.3,
  'temp_min': 16.88,
  'temp_max': 18.77,
  'pressure': 1022,
  'sea_level': 1022,
  'grnd_level': 1018,
  'humidity': 61,
  'temp_kf': 1.89,
  'dew_point': 11.1},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04n'}],
 'clouds': {'all': 92},
 'wind': {'speed': 1.37, 'deg': 336, 'gust': 2.48},
 'visibility': 10000,
 'pop': 0,
 'sys': {'pod': 'n'},
 'dt_txt': '2026-09-26 18:00:00'}

In [38]:
json['city']

{'id': 6545310,
 'name': 'Mitte',
 'coord': {'lat': 52.52, 'lon': 13.405},
 'country': 'DE',
 'population': 329078,
 'timezone': 7200,
 'sunrise': 1790398713,
 'sunset': 1790441820}

Gathering the important information from the data

In [39]:
json['list'][0]['dt_txt']

'2026-09-26 18:00:00'

In [40]:
json['list'][0]['main']['temp']

18.77

In [41]:
json['list'][0]['main']['feels_like']

18.3

In [42]:
json['list'][0]['weather'][0]['description']

'overcast clouds'

In [43]:
json['list'][0]['wind']['speed']

1.37

In [44]:
# check if it has rained in the last 3 hours if not enter 0
json['list'][0].get('rain', {}).get('3h', 0)

0

Creating a DataFrame

In [45]:
berlin_weathers= []

for item in json['list']:
    berlin_weather = {
        'forecast_time': item.get('dt_txt', None),
        'temperature': item['main'].get('temp', None),
        'feels_like': item['main'].get('feels_like', None),
        'forecast': item['weather'][0].get('description', None),
         'rain': item.get('rain', {}).get('3h', 0),
        'wind': item['wind'].get('speed', None)   
     }
    
    berlin_weathers.append(berlin_weather)

berlin_weather_df = pd.DataFrame(berlin_weathers)
berlin_weather_df

,forecast_time,temperature,feels_like,forecast,rain,wind
0,2026-09-26 18:00:00,18.77,18.30,overcast clouds,0,1.37
1,2026-09-26 21:00:00,13.96,13.32,broken clouds,0,1.79
2,2026-09-27 00:00:00,11.23,10.63,broken clouds,0,2.04
3,2026-09-27 03:00:00,10.20,9.60,clear sky,0,2.28
4,2026-09-27 06:00:00,12.67,12.21,broken clouds,0,3.04
5,2026-09-27 09:00:00,19.34,19.13,broken clouds,0,2.67
6,2026-09-27 12:00:00,21.52,20.98,few clouds,0,3.81
7,2026-09-27 15:00:00,21.68,21.21,clear sky,0,3.48
8,2026-09-27 18:00:00,18.87,18.56,clear sky,0,3.47
9,2026-09-27 21:00:00,15.94,15.57,clear sky,0,3.83


## Gathering weather data from SQL as well as putting it into a 'weathers' table for the cities

In [46]:
# Create a function to get the data from all three cities

def gather_weather_datas(cities):

   # Get the data and information from SQL

   schema = os.environ["schema_name"]
   host = os.environ["host_address"]
   user = os.environ["user_name"]
   password = os.environ["password"]
   port = os.environ["port_number"]

   connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

   # get the data from the 'cities' table
   cities_from_sql = pd.read_sql("cities", con=connection_string)
   cities_from_sql


   API_key = os.environ["openWeatherApi"]
   weathers= []


   for _, city in cities_from_sql.iterrows(): # iterrows returns index and a value, i dont need the index

      latitude = city["latitude"]
      longitude = city["longitude"]
      city_id = city["city_id"]

      url = (f"https://api.openweathermap.org/data/2.5/forecast?lat={latitude}&lon={longitude}&appid={API_key}&units=metric")
      response = requests.get(url)
      weather_data = response.json()

      # Creating a DataFrame from the data
      for item in weather_data['list']:
         weather = {
            "city_id": city_id,
            'forecast_time': item.get('dt_txt', None),
            'temperature': item['main'].get('temp', None),
            'feels_like': item['main'].get('feels_like', None),
            'forecast': item['weather'][0].get('description', None),
            'rain': item.get('rain', {}).get('3h', 0),
            'wind': item['wind'].get('speed', None)   
         } 
         weathers.append(weather)

   weather_df = pd.DataFrame(weathers)
   return weather_df

In [47]:
cities_weather_df = gather_weather_datas(["Berlin", "Hamburg", "Munich"])
cities_weather_df

,city_id,forecast_time,temperature,feels_like,forecast,rain,wind
0,1,2026-09-26 18:00:00,18.77,18.30,overcast clouds,0.00,1.37
1,1,2026-09-26 21:00:00,13.96,13.32,broken clouds,0.00,1.79
2,1,2026-09-27 00:00:00,11.23,10.63,broken clouds,0.00,2.04
3,1,2026-09-27 03:00:00,10.20,9.60,clear sky,0.00,2.28
4,1,2026-09-27 06:00:00,12.67,12.21,broken clouds,0.00,3.04
...,...,...,...,...,...,...,...
115,3,2026-10-01 03:00:00,17.99,18.40,light rain,2.55,1.00
116,3,2026-10-01 06:00:00,17.17,17.45,light rain,1.53,2.11
117,3,2026-10-01 09:00:00,19.18,19.29,overcast clouds,0.00,0.88
118,3,2026-10-01 12:00:00,21.25,21.18,overcast clouds,0.00,0.75


weather_df to 'weathers' table in SQL

In [48]:
cities_weather_df.to_sql('weathers',
                  if_exists='append',
                  con=connection_string,
                  index=False)

120

## Flight data using the AeroDataBoxAPI

### Airport API by location

Goal: to extract information on flights to our cities

- what does the API ned to work?
  - airport API needs lat, lon
  - flights API needs ICAO codes to work
- make the API call
  - first API call: I can make a first API call to get the airport codes
  - second API call: I can use the codes then to make the flights API call
- exploring and extracting the info I need from response json
- it will al lbe put into a DataFrame
- The DF gets send to the database in MySQL

Start with Airport API to get the airports based on location: \
First attempt with Berlin

In [49]:
load_dotenv()

headers = {
	"x-rapidapi-key": os.environ["x-rapidapi-key"],
	"x-rapidapi-host": os.environ["x-rapidapi-host"],
    "Content-Type": os.environ["Content-Type"]
}

url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"withFlightInfoOnly":"True","radiusKm":"50","limit":"10","lat":"52.31","lon":"13.24"}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'searchBy': {'lat': 52.31, 'lon': 13.24}, 'count': 2, 'items': [{'icao': 'EDDB', 'iata': 'BER', 'name': 'Berlin Brandenburg', 'shortName': 'Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDDT', 'iata': 'TXL', 'name': 'Berlin -Tegel', 'shortName': '-Tegel', 'municipalityName': 'Berlin', 'location': {'lat': 52.5597, 'lon': 13.287699}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}


Turning the API's data into a DataFrame with `.json_normalize()`

In [50]:
pd.json_normalize(response.json()['items'])

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
1,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699


Gathering the data for multiple cities, using their latitude and longitude

In [51]:
def get_airports(latitudes, longitudes):
    # API headers
    headers = {
	"x-rapidapi-key": os.environ["x-rapidapi-key"],
	"x-rapidapi-host": os.environ["x-rapidapi-host"],
    "Content-Type": os.environ["Content-Type"]
    }

    # DataFrame to store results
    all_airports = []

    for lat, lon in zip(latitudes, longitudes):
    # Construct the URL with the latitude and longitude
        url = url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"lat":lat,"lon":lon,"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)

    return pd.concat(all_airports, ignore_index=True)

Using the latitudes and longitudes from the citites table

In [52]:
# coordinates for Berlin, Hamburg, Munich
latitudes = [52.5200, 53.55, 50.9364]
longitudes = [13.4050, 10, 6.95278]

airports_table  = get_airports(latitudes, longitudes)
airports_table

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDK,CGN,Cologne Bonn,Bonn,Cologne,DE,Europe/Berlin,50.86590,7.142739
4,EDDL,DUS,Düsseldorf,Düsseldorf,Düsseldorf,DE,Europe/Berlin,51.28950,6.766779


#### Create the DataFrames for SQL

First add more tables to SQL:
```sql
-- Create 'weathers' table
CREATE TABLE weathers(
	weather_entry_id INT AUTO_INCREMENT NOT NULL, -- Automatically generated for each weather entry
    city_id INT NOT NULL, -- ID for each city
    forecast_time DATETIME, -- Date and time of the forecast
    temperature FLOAT, -- temperature of the forecast
    feels_like FLOAT, -- what the temperature feels like
    forecast VARCHAR(255), -- weather forecast
    rain FLOAT, -- whether it has rained or not and how much in mm
    wind FLOAT, -- wind speed
    PRIMARY KEY (weather_entry_id),
    FOREIGN KEY (city_id) REFERENCES cities(city_id) -- foreign key to connect weather data to cities
);

-- Create 'airports' table
CREATE TABLE airports (
	airport_icao VARCHAR(25), -- icao of the airport
    airport_name VARCHAR(255), -- the name of the airport
    PRIMARY KEY (airport_icao)
);

-- Create 'cities_airports' table
CREATE TABLE cities_airports(
	city_id INT, -- ID of the city
    airport_icao VARCHAR(5), -- icao of the city's airport
	FOREIGN KEY (city_id) REFERENCES cities(city_id), -- forgein key to connect cities_airports to cities
    FOREIGN KEY (airport_icao) REFERENCES airports(airport_icao) -- foreign key to connect cities_airports to airports
);

-- Create 'flights' table
CREATE TABLE flights (
	flight_id INT AUTO_INCREMENT NOT NULL, -- Automatically generated ID for each flight
    flight_number VARCHAR(25), -- the flight number, made of letters and numbers
    departure_airport_icao VARCHAR(25), -- icao of the airport the flight departed from
    arrival_airport_icao VARCHAR(25), -- icao of the airport the flight is arriving at
    scheduled_arrival_time DATETIME, -- the time when the flight arrives
    PRIMARY KEY (flight_id),
    FOREIGN KEY (arrival_airport_icao) REFERENCES airports(airport_icao) -- forgein key to connect flights to airports
);
```

In [53]:
# creating DataFrame 'airports'
airport_name = airports_table['name']
airport_icao = airports_table['icao']

airports = pd.DataFrame({'airport_icao': airport_icao,
                         'airport_name': airport_name})
airports

,airport_icao,airport_name
0,EDDT,Berlin -Tegel
1,EDDB,Berlin Brandenburg
2,EDDH,Hamburg
3,EDDK,Cologne Bonn
4,EDDL,Düsseldorf


In [54]:
# connecting the DataFrame with SQL
airports.to_sql('airports',
                if_exists='append',
                con=connection_string,
                index=False)

5

In [55]:
airports_from_sql = pd.read_sql("airports", con=connection_string)
airports_from_sql

,airport_icao,airport_name
0,EDDB,Berlin Brandenburg
1,EDDH,Hamburg
2,EDDK,Cologne Bonn
3,EDDL,Düsseldorf
4,EDDT,Berlin -Tegel


In [56]:
cities_from_sql

,city_id,city,latitude,longitude,country
0,1,Berlin,52.5200,13.40500,Germany
1,2,Hamburg,53.5500,10.00000,Germany
2,3,Cologne,50.9364,6.95278,Germany


In [57]:
# creating DataFrame 'cities_airports'
cities_airports = airports_table[['municipalityName', 'icao']].merge(
                                                                cities_from_sql[['city_id', 'city']],
                                                                left_on='municipalityName',
                                                                right_on='city',
                                                                how='left'
                                                                )
cities_airports

,municipalityName,icao,city_id,city
0,Berlin,EDDT,1.0,Berlin
1,Berlin,EDDB,1.0,Berlin
2,Hamburg,EDDH,2.0,Hamburg
3,Cologne,EDDK,3.0,Cologne
4,Düsseldorf,EDDL,NaN,NaN


In [58]:
cities_airports = cities_airports[['city_id', 'icao']]

cities_airports = cities_airports.rename(
    columns={'icao': 'airport_icao'}
)


In [59]:
cities_airports['city_id'] = cities_airports['city_id'].astype('Int64')
cities_airports

,city_id,airport_icao
0,1,EDDT
1,1,EDDB
2,2,EDDH
3,3,EDDK
4,<NA>,EDDL


Düsseldorf remains without an ID since it was not one of the three cities, but one within reach of Cologne. To keep it simple I'll drop it.

In [60]:
cities_airports = cities_airports.dropna(subset=['city_id'])
cities_airports

,city_id,airport_icao
0,1,EDDT
1,1,EDDB
2,2,EDDH
3,3,EDDK


In [61]:
# connecting the DataFrame with SQL
cities_airports.to_sql('cities_airports',
                        if_exists='append',
                        con=connection_string,
                        index=False)

4

In [62]:
cities_airports_from_sql = pd.read_sql("cities_airports", con=connection_string)
cities_airports_from_sql

,city_id,airport_icao
0,1,EDDT
1,1,EDDB
2,2,EDDH
3,3,EDDK


### Flight API for specific date (The next day)

In [63]:
# getting tomorrow's date

today_date = datetime.now()
tomorrow  = today_date + timedelta(days=1)

tomorrow_date = tomorrow.strftime('%Y-%m-%d')
tomorrow_date

'2026-09-27'

In [64]:
# first try using icao of Berlin Brandenburg

api_key = os.environ["x-rapidapi-key"]
icao = "EDDB"
time_1 = "00:00"
time_2 = "11:59" # remaining 12 hours of the day are missing. Hours should be more than beginning of the search range, but not more than by 12 hours per API call.

url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow_date}T{time_1}/{tomorrow_date}T{time_2}"

querystring = {"withLeg":"true", "direction":"Arrival", "withCancelled":"false", "withCodeshared":"true", "withCargo":"false", "withPrivate":"false"}

headers = {
	"x-rapidapi-key": os.environ["x-rapidapi-key"],
	"x-rapidapi-host": os.environ["x-rapidapi-host"],
	"Content-Type": os.environ["Content-Type"]
}
response = requests.request("GET",
                            url,
                            headers = headers,
                            params = querystring)

flights_json = response.json()

flights_json

{'arrivals': [{'departure': {'airport': {'icao': 'LTBJ',
     'iata': 'ADB',
     'name': 'İzmir',
     'countryCode': 'tr',
     'timeZone': 'Europe/Istanbul'},
    'scheduledTime': {'utc': '2026-09-27 01:00Z',
     'local': '2026-09-27 04:00+03:00'},
    'revisedTime': {'utc': '2026-09-27 01:00Z',
     'local': '2026-09-27 04:00+03:00'},
    'quality': ['Basic', 'Live']},
   'arrival': {'scheduledTime': {'utc': '2026-09-27 04:05Z',
     'local': '2026-09-27 06:05+02:00'},
    'revisedTime': {'utc': '2026-09-27 04:05Z',
     'local': '2026-09-27 06:05+02:00'},
    'terminal': '1',
    'gate': 'N01',
    'baggageBelt': 'B4',
    'quality': ['Basic', 'Live']},
   'number': 'XQ 966',
   'status': 'Expected',
   'codeshareStatus': 'IsOperator',
   'isCargo': False,
   'aircraft': {'reg': 'TC-SOR', 'modeS': '4BCDF2', 'model': 'Boeing 737-800'},
   'airline': {'name': 'Sun Express', 'iata': 'XQ', 'icao': 'SXS'}},
  {'departure': {'airport': {'icao': 'LTAU',
     'iata': 'ASR',
     'name': 

Data exploration

In [65]:
flights_json.keys()

dict_keys(['arrivals'])

In [66]:
flights_json['arrivals'][0]

{'departure': {'airport': {'icao': 'LTBJ',
   'iata': 'ADB',
   'name': 'İzmir',
   'countryCode': 'tr',
   'timeZone': 'Europe/Istanbul'},
  'scheduledTime': {'utc': '2026-09-27 01:00Z',
   'local': '2026-09-27 04:00+03:00'},
  'revisedTime': {'utc': '2026-09-27 01:00Z',
   'local': '2026-09-27 04:00+03:00'},
  'quality': ['Basic', 'Live']},
 'arrival': {'scheduledTime': {'utc': '2026-09-27 04:05Z',
   'local': '2026-09-27 06:05+02:00'},
  'revisedTime': {'utc': '2026-09-27 04:05Z',
   'local': '2026-09-27 06:05+02:00'},
  'terminal': '1',
  'gate': 'N01',
  'baggageBelt': 'B4',
  'quality': ['Basic', 'Live']},
 'number': 'XQ 966',
 'status': 'Expected',
 'codeshareStatus': 'IsOperator',
 'isCargo': False,
 'aircraft': {'reg': 'TC-SOR', 'modeS': '4BCDF2', 'model': 'Boeing 737-800'},
 'airline': {'name': 'Sun Express', 'iata': 'XQ', 'icao': 'SXS'}}

In [67]:
flights_json['arrivals'][0].keys()

dict_keys(['departure', 'arrival', 'number', 'status', 'codeshareStatus', 'isCargo', 'aircraft', 'airline'])

Looking at the content of the json, the important information is: \
    - flight number \
    - departure airport icao ['airport': 'icao'] \
    - arrival icao \
    - scheduled arrival time, local

#### Creating the flights DataFrame

In [68]:
flights_data = []

for item in flights_json['arrivals']:
    flight_data = {
        'flight_number': item.get('number', None),
        'departure_airport_icao': item['departure']['airport'].get('icao', None),
        'arrival_airport_icao': icao,
        'scheduled_arrival_time': item["arrival"]["scheduledTime"].get("local", None)
    }

    flights_data.append(flight_data)

flight_data_berlin_df = pd.DataFrame(flights_data)
flight_data_berlin_df.head()

,flight_number,departure_airport_icao,arrival_airport_icao,scheduled_arrival_time
0,XQ 966,LTBJ,EDDB,2026-09-27 06:05+02:00
1,XQ 1716,LTAU,EDDB,2026-09-27 06:45+02:00
2,HU 489,ZBAA,EDDB,2026-09-27 06:45+02:00
3,W6 7037,LZIB,EDDB,2026-09-27 07:15+02:00
4,W4 3109,LROP,EDDB,2026-09-27 07:15+02:00


Adjusting the `arrival_time` without the `+02:00`

In [69]:
flight_data_berlin_df["scheduled_arrival_time"] = flight_data_berlin_df["scheduled_arrival_time"].str[:-6]
flight_data_berlin_df.head()

,flight_number,departure_airport_icao,arrival_airport_icao,scheduled_arrival_time
0,XQ 966,LTBJ,EDDB,2026-09-27 06:05
1,XQ 1716,LTAU,EDDB,2026-09-27 06:45
2,HU 489,ZBAA,EDDB,2026-09-27 06:45
3,W6 7037,LZIB,EDDB,2026-09-27 07:15
4,W4 3109,LROP,EDDB,2026-09-27 07:15


#### Creating a function to get flight data from multiple cities

In [70]:
# Creating a function
def gather_flight_datas(icao_list):
    flight_data = []
    api_key = os.environ['RapidFlightAPIKey']
    today_date = datetime.now()
    tomorrow = today_date + timedelta(days=1)
    tomorrow_date = tomorrow.strftime('%Y-%m-%d')

    for icao in icao_list:
        times = [['00:00', '11:59'],    # need two start and end times since the API call can only get max 12 hours of data
                 ['12:00', '23:59']]

    for time in times:
        url = f'https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow_date}T{time[0]}/{tomorrow_date}T{time[1]}'

        querystring = {'withLeg':'true', 
                       'direction':'Arrival', 
                       'withCancelled':'false', 
                       'withCodeshared':'true', 
                       'withCargo':'false', 
                       'withPrivate':'false'
                       }

        headers = {'x-rapidapi-key': os.environ['x-rapidapi-key'],
                   'x-rapidapi-host': os.environ['x-rapidapi-host'],
                   'Content-Type': os.environ['Content-Type']}

        response = requests.request('GET', url, headers = headers, params = querystring)
        flights_json = response.json()

        for item in flights_json['arrivals']:
            flight_item ={'flight_number': item.get('number', None),
                          'departure_airport_icao': item['departure']['airport'].get('icao', None),
                          'arrival_airport_icao': icao,
                          'scheduled_arrival_time': item['arrival']['scheduledTime'].get('local',None)
                          }
        flight_data.append(flight_item)


    flight_data_df = pd.DataFrame(flight_data)
    flight_data_df["scheduled_arrival_time"] = flight_data_df["scheduled_arrival_time"].str.split('+').str[0]



    return flight_data_df

In [71]:
# Get the data from the three cities' airports
icao_list = ["EDDT", "EDDB", "EDDH", "EDDK"] 
gather_flight_datas(icao_list)

,flight_number,departure_airport_icao,arrival_airport_icao,scheduled_arrival_time
0,FR 5519,LEPA,EDDK,2026-09-27 11:55
1,XQ 116,LTAI,EDDK,2026-09-27 23:50


##### Sending the data from the DataFrame to SQL

In [72]:
flights = gather_flight_datas(icao_list)
flights

,flight_number,departure_airport_icao,arrival_airport_icao,scheduled_arrival_time
0,FR 5206,LMML,EDDK,2026-09-27 11:55
1,XQ 116,LTAI,EDDK,2026-09-27 23:50


In [73]:
flights.to_sql('flights',
               if_exists='append',
               con=connection_string,
               index=False)

2